
## Load the Datasets


In [ ]:
#import pandas as pd

import pandas as pd

In [ ]:
#Loads CityBikes data

bikes_url = "https://raw.githubusercontent.com/ljaycab/citybike-census/main/data/raw/citybike_networks_us.csv"

bikes_df = pd.read_csv(bikes_url)

bikes_df.head()

In [ ]:
#Load Census data

census_url = "https://raw.githubusercontent.com/ljaycab/citybike-census/main/data/raw/census_places.csv"

census_df = pd.read_csv(census_url, dtype={'geoid': str})

census_df.head()

# Check the CityBikes Data

In [ ]:
#Count how many cities are missing a state

bikes_df['state_abbr'].isna().sum()

In [ ]:
#Counts how many records each city has

bikes_df['city'].value_counts()

In [ ]:
#Show the two NY rocords

bikes_df[bikes_df['city'] == 'New York']

In [ ]:
#Create a copy of the CityBikes data

bikes_clean = bikes_df.copy()

print("Original records:", len(bikes_df))
print("Working records:", len(bikes_clean))

# Combine Data from Both APIs

In [ ]:
# Remove Lime networks (no dock counts)
bikes_clean = bikes_clean[bikes_clean['pct_missing_capacity'] == 0]

# Combine networks in the same city, adding only stations and docks
bikes_clean = bikes_clean.groupby(['city', 'state_abbr'], as_index=False).agg(
    n_stations=('n_stations', 'sum'),
    total_docks=('total_docks', 'sum')
)

bikes_clean.head()

In [ ]:
# State names
state_names = {
    'NY': 'New York', 'IL': 'Illinois', 'DC': 'District of Columbia',
    'CA': 'California', 'MA': 'Massachusetts', 'PA': 'Pennsylvania',
    'OR': 'Oregon', 'WI': 'Wisconsin', 'FL': 'Florida', 'HI': 'Hawaii',
    'CO': 'Colorado', 'TX': 'Texas', 'NE': 'Nebraska', 'MI': 'Michigan',
    'OH': 'Ohio', 'UT': 'Utah', 'IN': 'Indiana', 'TN': 'Tennessee',
    'AZ': 'Arizona', 'NV': 'Nevada', 'IA': 'Iowa', 'AR': 'Arkansas',
    'VA': 'Virginia', 'SC': 'South Carolina', 'ND': 'North Dakota',
}

In [ ]:
# Add a full state name column
bikes_clean['state'] = bikes_clean['state_abbr'].map(state_names)
bikes_clean.head()

# Merge the Datasets

In [ ]:
merged = bikes_clean.merge(census_df, on=['city', 'state'], how='left')

# Rows with no geoid did not match
merged[merged['geoid'].isna()]

In [ ]:
census_df[census_df['city'].str.contains('Honolulu|Indianapolis|Nashville|San Francisco', case=False)]

In [ ]:
# Make the bike city names match the Census names
bikes_clean = bikes_clean.replace({'city': {
    'San Francisco Bay Area': 'San Francisco',
    'Honolulu': 'Urban Honolulu',
    'Indianapolis': 'Indianapolis city (balance)',
    'Nashville': 'Nashville-Davidson metropolitan government (balance)',
}})

# Drop Kailua-Kona (too small to compare)
bikes_clean = bikes_clean[bikes_clean['city'] != 'Kailua-Kona']

bikes_clean.shape

In [ ]:
merged = bikes_clean.merge(census_df, on=['city', 'state'], how='left')

# Should show no rows if everything matched
merged[merged['geoid'].isna()]

In [ ]:
print("Cities:", len(merged))
print("Duplicate cities:", merged['geoid'].duplicated().sum())

# Count missing values in each column
merged.isna().sum()

In [ ]:
# Check data types
merged.dtypes

# Save to SQLite

In [ ]:
import sqlite3

# Cutoff: only cities with at least this many residents
MIN_POPULATION = 250000

major = merged[merged['population'] >= MIN_POPULATION]

# Open the database file (creates bike_share.db if it doesn't exist yet)
conn = sqlite3.connect("bike_share.db")

# Save the tables (all 51 cities stay in city_combined)
bikes_df.to_sql("citybike_networks", conn, if_exists="replace", index=False)
census_df.to_sql("census_places", conn, if_exists="replace", index=False)
merged.to_sql("city_combined", conn, if_exists="replace", index=False)
major.to_sql("city_major", conn, if_exists="replace", index=False)

# Check that everything saved
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
print(pd.read_sql("SELECT COUNT(*) AS all_cities FROM city_combined", conn))
print(pd.read_sql("SELECT COUNT(*) AS major_cities FROM city_major", conn))

conn.close()